In [ ]:
! pip install pims

! pip install trackpy

In [ ]:
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
df=pd.read_csv('UV_sweep/UV_sweep_1.csv', sep=",")


In [ ]:
i=1
df=pd.read_csv('UV_sweep/UV_sweep_'+str(i)+'.csv', sep=",")
#or
df=pd.read_csv(f'UV_sweep/UV_sweep_{i}.csv', sep=",")


In [ ]:
for i in range(0,10):
	df=pd.read_csv(f'UV_sweep/UV_sweep_{i}.csv', sep=",")


In [ ]:
for i in range(0,10):
	df=pd.read_csv(f'UV_sweep/UV_sweep_{i}.csv', sep=",")
	peak_x=df['x'][df['y'].argmax()]


In [ ]:
peak_positions=[]
peak_heights=[]
for i in range(0,100):
	df=pd.read_csv(f'UV_sweep/UV_sweep_{i}.csv', sep=",")
	peak_x=df['x'][df['y'].argmax()]
	peak_y=df['y'].max()
	peak_positions.append(peak_x)
	peak_heights.append(peak_y)


In [ ]:
np.asarray(peak_heights).argmax()
peak_positions[np.asarray(peak_heights).argmax()]

In [ ]:
plt.plot(peak_positions,peak_heights,'o')
plt.plot(peak_positions[np.asarray(peak_heights).argmax()],peak_heights[np.asarray(peak_heights).argmax()],'+')
plt.show()

In [ ]:
img=np.loadtxt('text_for_example.txt')


In [ ]:
img

In [ ]:
plt.imshow(img,'gray')
plt.show()



In [ ]:
plt.imshow(img>150)
plt.show()


In [ ]:
plt.imshow(img,'prism')
plt.show()

In [ ]:
img=plt.imread('Reading_shield.jpg', format='jpeg')
img.shape


In [ ]:
plt.imshow(img)
plt.show()

In [ ]:
plt.imshow(img[:,:,0],'gray')


In [ ]:
import trackpy as tp

import pims


In [ ]:
red_channel = pims.open('BBBC015_v1_images/*1.JPG')
green_channel = pims.open('BBBC015_v1_images/*0.JPG')


In [ ]:
def gray(image):
    return image[:, :, 1]


In [ ]:
gray(red_channel[0]*2)


In [ ]:
gray(green_channel[0])


In [ ]:
gray(red_channel[0]) +gray(green_channel[0])


In [ ]:
combinedRGB=np.zeros( red_channel[0].shape)
combinedRGB[:,:,0]= red_channel[0][:,:,0]
combinedRGB[:,:,1]= green_channel[0][:,:,0]


In [ ]:
plt.imshow(combinedRGB/100)
plt.show()

In [ ]:
dfr = tp.locate(gray(red_channel[0]), 61, invert=False)


In [ ]:


tp.annotate(dfr, 5*gray(red_channel[0]),plot_style={'markersize': 10})


In [ ]:
image=sp.ndimage.uniform_filter(gray(red_channel[0]),size=10)
dfr = tp.locate(image, 61, invert=False)
tp.annotate(dfr, 5*image)

In [ ]:
plt.hist(dfr['mass'], bins=20)
plt.xlabel('mass')
plt.ylabel('count')
plt.show()


In [ ]:
df = tp.locate(image, 61, minmass=14000, invert=False)


In [ ]:
tp.annotate(df, 5*image)

In [ ]:
dfg = tp.locate(gray(green_channel[0]), 5, minmass=0, invert=False)


In [ ]:
tp.annotate(dfg, 5*gray(green_channel[0]),plot_style={'markersize': 2})

In [ ]:
dfg = tp.locate(gray(green_channel[0]), 5, minmass=70, invert=False)


In [ ]:
tp.annotate(dfg, 2*gray(green_channel[0]),plot_style={'markersize': 2})

In [ ]:
from scipy.spatial import KDTree
df1 = dfr[['x', 'y']] #x and y from red
tree = KDTree(df1)
df2=dfg[['x', 'y']] #x and y from green
tree2 = KDTree(df2)

In [ ]:
distances, ids=tree.query(df2, k=1)


In [ ]:
unique_ids,ids_counts=np.unique(ids,return_index=False, return_counts=True)
ids_counts_all=np.append(ids_counts,np.zeros(len(df1)-len(ids_counts)))
plt.hist(ids_counts_all,bins=30)
plt.show()

In [ ]:
df1 = dfr[['x', 'y']]

df2=dfg[['x', 'y']]
tree2 = KDTree(df2)
distances, ids=tree2.query(df1, k=1)

In [ ]:
plt.hist(distances,bins=20)
plt.show()

In [ ]:
ids_counts_all_frames=[]
distances_all_frames=[]
for i in range(len(green_channel)):
    image=2*sp.ndimage.uniform_filter(gray(green_channel[i]),size=3)
    dfg = tp.locate(image, 5, minmass=70, invert=False)
    image=2*sp.ndimage.uniform_filter(gray(red_channel[i]),size=30)
    dfr = tp.locate(image, 61, minmass=14000, invert=False)
    #however, in some cases minmass will be inappropriate for the image given - requires try and except or similar
    df1 = dfr[['x', 'y']]
    tree = KDTree(df1)
    df2=dfg[['x', 'y']]
    tree2 = KDTree(df2)
    distances, ids=tree.query(df2, k=1)
    distances2, ids2=tree2.query(df1, k=1)
    unique_ids,ids_counts=np.unique(ids,return_index=False, return_counts=True)
    ids_counts_all=np.append(ids_counts,np.zeros(len(df1)-len(ids_counts)))
    ids_counts_all_frames=np.append(ids_counts_all_frames,ids_counts_all)
    distances_all_frames=np.append(distances_all_frames,distances2)